# AV-Forensics — Balanced Retraining Notebook

**Purpose:** Resume training from the existing 2-epoch checkpoint with:
- ✅ Balanced `WeightedRandomSampler` (equal RVRA/RVFA/FVRA/FVFA)
- ✅ 5 more epochs with lower LR (fine-tuning)
- ✅ Unfreeze audio encoder for the last 2 epochs
- ✅ Threshold calibrated on balanced val set (not Youden's J)
- ✅ Exports new `model.pth` + `config.json`

**Before you start:**
1. Runtime → Change runtime type → **T4 GPU**
2. All 14,851 `.npy` cache files must already be in Drive (`avforensics_cache/`)
3. `best_model.pth` must be in Drive (`avforensics_checkpoints/`)


## Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR  = '/content/drive/MyDrive/avforensics_checkpoints'
CACHE_DIR = '/content/drive/MyDrive/avforensics_cache'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Drive mounted.')
print('Checkpoint dir:', SAVE_DIR)
print('Cache dir     :', CACHE_DIR)
ckpt = os.path.join(SAVE_DIR, 'best_model.pth')
if os.path.exists(ckpt):
    size_mb = os.path.getsize(ckpt) / 1e6
    print(f'✓ best_model.pth found ({size_mb:.1f} MB)')
else:
    print('✗ best_model.pth NOT found — check your Drive path')

## Cell 2 — Install Libraries

In [ ]:
!pip install -q -U numpy transformers mediapipe librosa soundfile torchinfo
!apt-get install -qq ffmpeg
print('Libraries ready.')

## Cell 3 — Upload ML Scripts & Config

In [ ]:
import os, sys
ML_DIR = '/content/avforensics/ml'
os.makedirs(ML_DIR, exist_ok=True)

if not os.path.exists(os.path.join(ML_DIR, 'model.py')):
    print('Upload ml_scripts.zip (zip of the ml/ folder):')
    from google.colab import files
    import zipfile
    uploaded = files.upload()
    for fname in uploaded.keys():
        if fname.endswith('.zip'):
            with zipfile.ZipFile(fname, 'r') as z:
                z.extractall(ML_DIR)
            print('Extracted to', ML_DIR)
else:
    print('ML scripts already present.')

if ML_DIR not in sys.path:
    sys.path.insert(0, ML_DIR)

from config import (
    AUDIO_MODEL_NAME, CLASSES, CLASS_TO_ID, BINARY_LABEL, AUDIO_FAKE, VIDEO_FAKE,
    SR, N_FRAMES, IMG_SIZE, AUDIO_LEN, WINDOW_SEC, STRIDE_SEC,
    EMBED_DIM, LAMBDA_SYNC, AUX_WEIGHT, BATCH_SIZE,
)
print('Config loaded.')
print(f'Audio model : {AUDIO_MODEL_NAME}')
print(f'Classes     : {CLASSES}')

## Cell 4 — Build Balanced DataLoaders

> **Key fix:** `WeightedRandomSampler` ensures each training batch sees equal proportions of all 4 classes, eliminating the 75%-FAKE imbalance that caused the binary head bias.

In [ ]:
import glob, random
import numpy as np
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler
from dataset import scan_split, identity_grouped_split, AVDataset

cache_items = []
for cls in CLASSES:
    found_cls = []
    for split in ['train', 'test']:
        found_cls.extend(glob.glob(f'{CACHE_DIR}/{cls}/{split}/*_frames.npy'))
    if not found_cls:
        found_cls.extend(glob.glob(f'{CACHE_DIR}/**/{cls}/**/*_frames.npy', recursive=True))
    for ff in found_cls:
        af = ff.replace('_frames.npy', '_audio.npy')
        if os.path.exists(af):
            cache_items.append((ff, cls))

print(f'Total cached items: {len(cache_items)}')
for cls in CLASSES:
    n = sum(1 for _, c in cache_items if c == cls)
    print(f'  {cls}: {n}')

random.seed(42)
random.shuffle(cache_items)
val_frac  = 0.12
n_val     = int(len(cache_items) * val_frac)

by_class = {cls: [] for cls in CLASSES}
for item in cache_items:
    by_class[item[1]].append(item)

val_items, train_items = [], []
for cls, items in by_class.items():
    n_cls_val = max(1, int(len(items) * val_frac))
    val_items   += items[:n_cls_val]
    train_items += items[n_cls_val:]

random.shuffle(train_items)
print(f'\nTrain: {len(train_items)} | Val: {len(val_items)}')
    
class_counts = np.array([sum(1 for _, c in train_items if c == cls) for cls in CLASSES])
class_weights_arr = 1.0 / np.maximum(class_counts, 1)
sample_weights = np.array([
    class_weights_arr[CLASS_TO_ID[cls]] for _, cls in train_items
])
sampler = WeightedRandomSampler(
    weights=torch.from_numpy(sample_weights).float(),
    num_samples=len(train_items),
    replacement=True,
)

print('Class weights for sampler:', {cls: round(float(class_weights_arr[i]), 4) for i, cls in enumerate(CLASSES)})

class CachedAVDataset(torch.utils.data.Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        frames_path, cls_name = self.items[idx]
        audio_path = frames_path.replace('_frames.npy', '_audio.npy')
        frames = torch.from_numpy(np.load(frames_path).astype(np.float32))
        audio  = torch.from_numpy(np.load(audio_path).astype(np.float32))
        label_4    = CLASS_TO_ID[cls_name]
        label_bin  = BINARY_LABEL[cls_name]
        label_af   = AUDIO_FAKE[cls_name]
        label_vf   = VIDEO_FAKE[cls_name]
        return {
            'frames':    frames,
            'audio':     audio,
            'label_4':   torch.tensor(label_4,   dtype=torch.long),
            'label_bin': torch.tensor(label_bin, dtype=torch.long),
            'label_af':  torch.tensor(label_af,  dtype=torch.long),
            'label_vf':  torch.tensor(label_vf,  dtype=torch.long),
        }

train_ds = CachedAVDataset(train_items)
val_ds   = CachedAVDataset(val_items)

BATCH_SIZE = 6
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True)

print('DataLoaders ready.')

print('\nChecking sampler balance (first 4 batches):')
counts = {cls: 0 for cls in CLASSES}
for i, batch in enumerate(train_loader):
    if i >= 4: break
    for lbl in batch['label_4'].tolist():
        counts[CLASSES[lbl]] += 1
print(counts)

## Cell 5 — Load Model from Checkpoint

In [ ]:
import torch
from model import AVDeepfakeModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

model = AVDeepfakeModel(
    embed_dim=EMBED_DIM,
    audio_model=AUDIO_MODEL_NAME,
    freeze_cnn=False,
    freeze_layers=0,
).to(device)

ckpt_path = os.path.join(SAVE_DIR, 'best_model.pth')
state = torch.load(ckpt_path, map_location=device)

if 'audio_enc.layer_weights' in state:
    ckpt_shape = state['audio_enc.layer_weights'].shape[0]
    model_shape = model.audio_enc.layer_weights.shape[0]
    if ckpt_shape != model_shape:
        print(f'Adjusting layer_weights: {model_shape} → {ckpt_shape}')
        model.audio_enc.layer_weights = torch.nn.Parameter(
            torch.ones(ckpt_shape, device=device)
        )
        
missing, unexpected = model.load_state_dict(state, strict=False)
print(f'Loaded checkpoint from {ckpt_path}')
if missing:    print(f'  Missing keys  : {missing[:5]}')
if unexpected: print(f'  Unexpected keys: {unexpected[:5]}')

n_total     = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Params — total: {n_total:,} | trainable: {n_trainable:,}')

## Cell 6 — Fine-Tune Training (5 Epochs, Balanced)

Training strategy:
- **Epochs 1–3:** Lower LR `5e-5`, audio encoder partially frozen (layers 0–4)
- **Epochs 4–5:** Even lower LR `1e-5`, all layers unfrozen
- **Loss:** Same multi-task loss (binary + 4-class + audio_fake + video_fake + sync)
- **Class weights:** Applied to 4-class CE loss so minority class RVRA gets higher gradient


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.metrics import f1_score, accuracy_score

w_bin = torch.tensor([1.0, 1.0], dtype=torch.float32).to(device)
w4_tensor = torch.tensor([1.0, 1.0, 1.0, 1.0], dtype=torch.float32).to(device)
print('Balanced training active: batches sampled equally across all 4 classes.')

PHASE1_EPOCHS = 3
PHASE2_EPOCHS = 2
TOTAL_EPOCHS  = PHASE1_EPOCHS + PHASE2_EPOCHS
GRAD_ACCUM    = 4

def get_optimizer(lr):
    return torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                              lr=lr, weight_decay=1e-2)

def freeze_audio_layers(n_freeze):
    """Freeze first n_freeze transformer layers of the audio encoder."""
    if hasattr(model.audio_enc, 'backbone') and hasattr(model.audio_enc.backbone, 'encoder'):
        for i, layer in enumerate(model.audio_enc.backbone.encoder.layers):
            for p in layer.parameters():
                p.requires_grad = (i >= n_freeze)
    elif hasattr(model.audio_enc, 'backbone') and hasattr(model.audio_enc.backbone, 'layers'):
        for i, layer in enumerate(model.audio_enc.backbone.layers):
            for p in layer.parameters():
                p.requires_grad = (i >= n_freeze)

def train_epoch(loader, optimizer, scaler, epoch):
    model.train()
    total_loss, all_bin_true, all_bin_pred = 0.0, [], []
    optimizer.zero_grad()

    pbar = tqdm(loader, desc=f'Epoch {epoch} Train')
    for step, batch in enumerate(pbar):
        frames    = batch['frames'].to(device)
        audio     = batch['audio'].to(device)
        lbl_bin   = batch['label_bin'].to(device)
        lbl_4     = batch['label_4'].to(device)
        lbl_af    = batch['label_af'].to(device)
        lbl_vf    = batch['label_vf'].to(device)

        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            out = model(frames, audio)

            loss_bin  = F.cross_entropy(out['logits_bin'], lbl_bin, weight=w_bin)
            loss_4    = F.cross_entropy(out['logits_4'],   lbl_4,   weight=w4_tensor)
            loss_af   = F.cross_entropy(out['logits_afake'], lbl_af)
            loss_vf   = F.cross_entropy(out['logits_vfake'], lbl_vf)

            v_n = F.normalize(out['vis_emb'], dim=-1)
            a_n = F.normalize(out['aud_emb'], dim=-1)
            sync_scores = (v_n * a_n).sum(-1).mean(-1)
            sync_target = (1 - lbl_bin.float())
            loss_sync   = F.mse_loss(sync_scores.sigmoid(), sync_target)

            loss = (loss_bin
                    + 0.5  * loss_4
                    + 0.3  * loss_af
                    + 0.3  * loss_vf
                    + LAMBDA_SYNC * loss_sync) / GRAD_ACCUM

        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * GRAD_ACCUM
        pred_bin = out['logits_bin'].argmax(1).cpu().tolist()
        all_bin_true.extend(lbl_bin.cpu().tolist())
        all_bin_pred.extend(pred_bin)
        pbar.set_postfix(loss=f'{total_loss/(step+1):.4f}')

    acc = accuracy_score(all_bin_true, all_bin_pred)
    f1  = f1_score(all_bin_true, all_bin_pred, average='macro', zero_division=0)
    return total_loss / len(loader), acc, f1

@torch.no_grad()
def val_epoch(loader, epoch):
    model.eval()
    total_loss, all_bin_true, all_bin_pred, all_4_true, all_4_pred = 0.0, [], [], [], []
    all_bin_prob = []

    for batch in tqdm(loader, desc=f'Epoch {epoch} Val '):
        frames  = batch['frames'].to(device)
        audio   = batch['audio'].to(device)
        lbl_bin = batch['label_bin'].to(device)
        lbl_4   = batch['label_4'].to(device)
        lbl_af  = batch['label_af'].to(device)
        lbl_vf  = batch['label_vf'].to(device)

        out = model(frames, audio)
        loss_bin = F.cross_entropy(out['logits_bin'], lbl_bin, weight=w_bin)
        loss_4   = F.cross_entropy(out['logits_4'],   lbl_4,   weight=w4_tensor)
        total_loss += (loss_bin + 0.5 * loss_4).item()

        prob_bin = F.softmax(out['logits_bin'], 1)[:, 1].cpu().tolist()
        all_bin_prob.extend(prob_bin)
        all_bin_true.extend(lbl_bin.cpu().tolist())
        all_bin_pred.extend(out['logits_bin'].argmax(1).cpu().tolist())
        all_4_true.extend(lbl_4.cpu().tolist())
        all_4_pred.extend(out['logits_4'].argmax(1).cpu().tolist())

    acc_bin = accuracy_score(all_bin_true, all_bin_pred)
    f1_bin  = f1_score(all_bin_true, all_bin_pred, average='macro', zero_division=0)
    acc_4   = accuracy_score(all_4_true, all_4_pred)
    f1_4    = f1_score(all_4_true, all_4_pred, average='macro', zero_division=0)

    return {
        'loss':     total_loss / len(loader),
        'acc_bin':  acc_bin,
        'f1_bin':   f1_bin,
        'acc_4':    acc_4,
        'f1_4':     f1_4,
        'bin_prob': all_bin_prob,
        'bin_true': all_bin_true,
    }

scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))
best_f1     = 0.0
best_ckpt   = os.path.join(SAVE_DIR, 'balanced_best.pth')
history     = []

for epoch in range(1, TOTAL_EPOCHS + 1):
    if epoch <= PHASE1_EPOCHS:
        freeze_audio_layers(4)
        optimizer = get_optimizer(5e-5)
    else:
        freeze_audio_layers(0)
        optimizer = get_optimizer(1e-5)

    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'\n─── Epoch {epoch}/{TOTAL_EPOCHS} (trainable params: {n_trainable:,}) ───')

    tr_loss, tr_acc, tr_f1 = train_epoch(train_loader, optimizer, scaler, epoch)
    val_metrics             = val_epoch(val_loader, epoch)

    row = {
        'epoch':    epoch,
        'tr_loss':  tr_loss,
        'tr_acc':   tr_acc,
        'tr_f1':    tr_f1,
        **{k: v for k, v in val_metrics.items() if k not in ('bin_prob', 'bin_true')}
    }
    history.append(row)

    print(f'  Train — loss={tr_loss:.4f}  acc={tr_acc:.4f}  f1={tr_f1:.4f}')
    print(f'  Val   — loss={val_metrics["loss"]:.4f}  acc_bin={val_metrics["acc_bin"]:.4f}  '
          f'f1_bin={val_metrics["f1_bin"]:.4f}  acc_4={val_metrics["acc_4"]:.4f}  f1_4={val_metrics["f1_4"]:.4f}')

    if val_metrics['f1_bin'] > best_f1:
        best_f1 = val_metrics['f1_bin']
        torch.save(model.state_dict(), best_ckpt)
        print(f'  ✓ New best val F1-bin = {best_f1:.4f} — saved to {best_ckpt}')

    epoch_ckpt = os.path.join(SAVE_DIR, f'balanced_epoch_{epoch:02d}_f1_{val_metrics["f1_bin"]:.4f}.pth')
    torch.save(model.state_dict(), epoch_ckpt)

print(f'\nTraining complete. Best val F1-bin = {best_f1:.4f}')

## Cell 7 — Calibrate Threshold on Balanced Val Set

> Youden's J gave `0.9959` because the val set was 75% FAKE. Now the val set is balanced, so the threshold will be more representative.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, classification_report
import matplotlib.pyplot as plt

state = torch.load(best_ckpt, map_location=device)
model.load_state_dict(state, strict=False)
model.eval()
print('Loaded best balanced checkpoint.')

val_metrics = val_epoch(val_loader, epoch='final')
y_true = np.array(val_metrics['bin_true'])
y_prob = np.array(val_metrics['bin_prob'])

fpr, tpr, thresholds = roc_curve(y_true, y_prob)
j_idx        = np.argmax(tpr - fpr)
best_thresh  = float(thresholds[j_idx])
roc_auc      = roc_auc_score(y_true, y_prob)

print(f'ROC-AUC          : {roc_auc:.4f}')
print(f'Best threshold   : {best_thresh:.4f}  (Youden J = {tpr[j_idx]-fpr[j_idx]:.4f})')

y_pred = (y_prob > best_thresh).astype(int)
print('\nClassification Report (binary):')
print(classification_report(y_true, y_pred, target_names=['REAL', 'FAKE']))

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.4f}')
plt.scatter(fpr[j_idx], tpr[j_idx], color='red', zorder=5,
            label=f'Threshold = {best_thresh:.4f}')
plt.plot([0,1],[0,1],'k--')
plt.xlabel('FPR'); plt.ylabel('TPR')
plt.title('ROC Curve (Balanced Val Set)')
plt.legend(); plt.tight_layout()
plt.savefig('/content/drive/MyDrive/avforensics_plots/roc_balanced.png', dpi=120)
plt.show()

## Cell 8 — Export model.pth + config.json

In [ ]:
import json

EXPORT_DIR = '/content/drive/MyDrive/avforensics_export'
os.makedirs(EXPORT_DIR, exist_ok=True)

MODEL_OUT  = os.path.join(EXPORT_DIR, 'model.pth')
CONFIG_OUT = os.path.join(EXPORT_DIR, 'config.json')

torch.save(model.state_dict(), MODEL_OUT)

config = {
    'audio_model':    AUDIO_MODEL_NAME,
    'embed_dim':      EMBED_DIM,
    'n_frames':       N_FRAMES,
    'img_size':       IMG_SIZE,
    'sr':             SR,
    'audio_len':      AUDIO_LEN,
    'window_sec':     WINDOW_SEC,
    'stride_sec':     STRIDE_SEC,
    'fake_threshold': round(best_thresh, 4),
    'classes':        CLASSES,
    'metrics': {
        'val_roc_auc':   round(roc_auc, 4),
        'val_f1_binary': round(float(val_metrics['f1_bin']), 4),
        'val_acc_binary': round(float(val_metrics['acc_bin']), 4),
        'val_f1_4class':  round(float(val_metrics['f1_4']),   4),
        'val_acc_4class': round(float(val_metrics['acc_4']),  4),
        'training_epochs': TOTAL_EPOCHS,
        'balanced_sampler': True,
    }
}

with open(CONFIG_OUT, 'w') as f:
    json.dump(config, f, indent=2)

print(f'✓ model.pth  → {MODEL_OUT}  ({os.path.getsize(MODEL_OUT)/1e6:.1f} MB)')
print(f'✓ config.json → {CONFIG_OUT}')
print('\nconfig.json contents:')
print(json.dumps(config, indent=2))

## Cell 9 — Download to Local Machine

After running this cell, two files download to your browser:
- `model.pth` — replace `backend/model.pth`
- `config.json` — replace `backend/config.json`

Then restart the backend server.

In [ ]:
from google.colab import files
files.download(MODEL_OUT)
files.download(CONFIG_OUT)
print('Downloads started. Save both files to project root or backend directory.')


## Cell 10 — Quick Smoke Test (Upload a Video)

Upload one real and one fake video to verify the new model works before replacing your backend files.

In [ ]:
from google.colab import files
from preprocessing import preprocess_video
import torch.nn.functional as F

print('Upload a video to test (real or fake):')
uploaded = files.upload()
video_path = list(uploaded.keys())[0]

model.eval()
result = preprocess_video(video_path, N_FRAMES, IMG_SIZE, AUDIO_LEN, SR)
if not result['ok']:
    print('ERROR:', result['error'])
else:
    with torch.no_grad():
        frames = torch.from_numpy(result['frames']).unsqueeze(0).to(device)
        audio  = torch.from_numpy(result['audio']).unsqueeze(0).to(device)
        out    = model(frames, audio)

    prob_bin  = F.softmax(out['logits_bin'], 1)[0]
    prob_4    = F.softmax(out['logits_4'],   1)[0]
    pred_cls  = CLASSES[prob_4.argmax().item()]
    fake_prob = prob_bin[1].item()
    verdict   = 'FAKE' if fake_prob > best_thresh else 'REAL'

    v_n = F.normalize(out['vis_emb'][0], dim=-1)
    a_n = F.normalize(out['aud_emb'][0], dim=-1)
    sync = (v_n * a_n).sum(-1).cpu().numpy()

    print(f'\nVerdict     : {verdict}')
    print(f'Fake prob   : {fake_prob:.4f}  (threshold={best_thresh:.4f})')
    print(f'4-class     : {pred_cls} ({prob_4.tolist()})')
    print(f'Mean sync   : {sync.mean():.4f}')

    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 3))
    plt.plot(sync, marker='o', markersize=3)
    plt.axhline(0.5, color='orange', linestyle='--', label='0.5')
    plt.axhline(0.0, color='red', linestyle='--', label='0.0')
    plt.title(f'Sync Timeline — {verdict} ({pred_cls})  fake_prob={fake_prob:.3f}')
    plt.ylabel('Cosine similarity'); plt.xlabel('Frame')
    plt.legend(); plt.tight_layout(); plt.show()